# 11 · HatEval: más datos etiquetados del mismo esquema

**Reglas (fijadas antes de calcular nada):**
- ningún modelo preentrenado; todo se entrena aquí;
- todo lo que usa etiquetas queda dentro de la CV;
- HatEval entra **solo en los folds de entrenamiento**. La validación usa siempre **nuestras** filas y nuestros folds (`folds.csv`; `folds_seed2` solo confirma). Esto incluye la selección de hiperparámetros anidada y el *early stopping* de las redes;
- el stacker y los umbrales se ajustan solo con las OOF de nuestras filas;
- todo ensamble sigue el protocolo del 07 y lleva la regla de comillas (D34).

**Ejecución: ≈6–7 h la primera vez, reanudable.**
- reentrenar los 14 modelos del pool v2 con HatEval, con 2 pesos y en los 2 conjuntos de folds (≈4.5 h; lo más lento es la CV anidada de la configuración por etiqueta, ≈18 min por CV);
- miembros nuevos (≈30 min);
- modelos finales (≈50 min).

Cada fold se guarda en `models/11_hateval/cache/` en cuanto termina: si se interrumpe, al volver a ejecutar se sigue donde quedó. `RECOMPUTE = True` borra esa caché.

## 0. Configuración

In [1]:
import json
import os
import platform
import shutil
import time
import warnings

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import sklearn
from IPython.display import display
from joblib import Parallel, delayed

import ensemble as en
import external as ex
import modeling as md
import preprocessing as pp
from common import LABELS, PROCESSED_DIR, SEED, load_test, load_train, make_folds

RECOMPUTE = False  # True: borra la caché del 11 y recalcula todo (≈6–7 h). False: reutiliza lo ya calculado
RECOMPUTE = RECOMPUTE or os.environ.get("TWEETTOXIC_RECOMPUTE") == "1"
warnings.filterwarnings("ignore", category=UserWarning, module="torch")
pd.set_option("display.width", 250)
pd.set_option("display.max_colwidth", 110)
pd.set_option("display.max_columns", 40)
N_JOBS, FOLD_THREADS, FINAL_JOBS, FINAL_THREADS = 5, 2, 5, 2
OUT_DIR = md.model_folder("11")
CACHE_DIR = OUT_DIR / "cache"
if RECOMPUTE and CACHE_DIR.exists():
    shutil.rmtree(CACHE_DIR)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
CACHE_07, CACHE_09, CACHE_10 = (md.MODELS_DIR / f / "cache" for f in ("07_ensamble", "09_mejoras", "10_candidatos"))
P_COLS = [f"p_{lab}" for lab in LABELS]
AG = LABELS.index("AG")

train, test = load_train(), load_test()
Y = train[LABELS].to_numpy()
texts = train["text"]
folds = make_folds(train).to_numpy()
folds2 = make_folds(train, seed=SEED + 1, path=PROCESSED_DIR / "folds_seed2.csv").to_numpy()
FOLDSETS = {"folds": folds, "folds_seed2": folds2}
pp.load_lemma_cache(PROCESSED_DIR / "lemma_cache.parquet")
EMPTY_TR = md.empty_mask(texts)
QUOTED, QUOTED_TE = md.quoted_mask(texts), md.quoted_mask(test["text"])
REG = pd.read_csv(md.REGISTRY_PATH).set_index("model_id")
print(f"Python {platform.python_version()} | scikit-learn {sklearn.__version__} | SEED = {SEED} | RECOMPUTE = {RECOMPUTE}")
print(f"Train {len(train)} | test {len(test)}")

Python 3.14.6 | scikit-learn 1.9.1 | SEED = 42 | RECOMPUTE = False
Train 1731 | test 1154


## 1. Descarga y copia local

**Fuente:** el dataset `valeriobasile/HatEval` de Hugging Face, en una **revisión fija** (el *sha* del commit). Así, volver a ejecutar el notebook descarga exactamente los mismos archivos aunque el dataset cambie después.
- Es de acceso restringido: hay que aceptar sus términos en Hugging Face e iniciar sesión (`hf auth login`; antes `huggingface-cli login`). El notebook usa el token guardado por esa sesión; no lo pide, no lo imprime y no lo guarda.
- `huggingface_hub` baja los parquet originales y los copia **sin modificar** a `data/external/hateval/<split>.parquet`.
- `datasets` carga el mismo dataset en la misma revisión para comprobar que el número de filas coincide.
- `manifest.json` registra la revisión, la fecha, las filas y el sha256 de cada archivo. Si los archivos ya están y su sha256 coincide, no se descarga nada.

`data/external/` está en `.gitignore`: el corpus no se versiona (su licencia no permite redistribuirlo con fines comerciales y no hace falta: se reconstruye con esta celda). El `README.md` explica cómo obtenerlo.

**Cita y licencia** (también en `PROGRESO.md` §9):
> Basile, V., Bosco, C., Fersini, E., Nozza, D., Patti, V., Rangel Pardo, F. M., Rosso, P. y Sanguinetti, M. (2019). *SemEval-2019 Task 5: Multilingual Detection of Hate Speech Against Immigrants and Women in Twitter*. Proceedings of the 13th International Workshop on Semantic Evaluation (SemEval-2019), 54–63. Licencia **CC BY-NC 4.0**.

In [2]:
MANIFEST = ex.download_hateval()
print(f"{MANIFEST['repo']} @ {MANIFEST['revision']} | licencia {MANIFEST['license']} | acceso: {MANIFEST['gated']} | "
      f"descargado {MANIFEST['downloaded']} | {'reutilizado (sha256 verificado)' if MANIFEST['reused'] else 'descarga nueva'}")
display(pd.DataFrame(MANIFEST["files"]).T.assign(sha256=lambda d: d["sha256"].str[:16] + "…"))

valeriobasile/HatEval @ d170b5cb1844f3614493fce7c7eaaae087a5201c | licencia cc-by-nc-4.0 | acceso: auto | descargado 2026-10-16T01:39:24 | reutilizado (sha256 verificado)


,file,source,rows,rows_es,bytes,sha256
train,train.parquet,data/train-00000-of-00001.parquet,13500,4500,1510223,60366470d52e6d82…
dev,dev.parquet,data/dev-00000-of-00001.parquet,1500,500,183968,3ad7fcf9b6c89599…
test,test.parquet,data/test-00000-of-00001.parquet,4570,1599,496064,c492b18dec54fc9a…


## 2. Esquema e idioma

Se lee solo el **esquema** de los parquet y, de los datos, únicamente `id`, `text` y `language`. Las columnas de etiquetas existen, pero **no se cargan todavía**.

HatEval tiene tuits en inglés y en español; se usa solo el español.

In [3]:
schema = pq.read_schema(ex.HATEVAL_DIR / "train.parquet")
print("Columnas:", ", ".join(f"{f.name} ({f.type})" for f in schema))
H = ex.read_texts(language="es")
lang = pd.concat([pd.read_parquet(ex.HATEVAL_DIR / f"{s}.parquet", columns=["language"]).assign(split=s) for s in ex.SPLITS],
                 ignore_index=True)
display(pd.crosstab(lang["split"], lang["language"], margins=True, margins_name="total"))
assert H["id"].is_unique
print(f"Español: {len(H)} tuits | ids únicos: {H['id'].is_unique} | "
      f"ids de nuestro train en HatEval: {train['id'].isin(H['id']).sum()} de {len(train)} | "
      f"de nuestro test: {test['id'].isin(H['id']).sum()} de {len(test)}")
origin = np.select([H["id"].isin(train["id"]), H["id"].isin(test["id"])], ["id de nuestro train", "id de nuestro test"], "id nuevo")
display(pd.crosstab(H["split"], origin, margins=True, margins_name="total"))

Columnas: id (int64), text (string), target (string), language (string), HS (int64), TR (int64), AG (int64), __index_level_0__ (int64)


language,en,es,total
split,,,
dev,1000,500,1500
test,2971,1599,4570
train,9000,4500,13500
total,12971,6599,19570


Español: 6599 tuits | ids únicos: True | ids de nuestro train en HatEval: 1731 de 1731 | de nuestro test: 1154 de 1154


col_0,id de nuestro test,id de nuestro train,id nuevo,total
split,,,,
dev,113,173,214,500
test,0,0,1599,1599
train,1041,1558,1901,4500
total,1154,1731,3714,6599


**Interpretación:**
- **HatEval en español tiene 6599 tuits:** 4500 en su train, 500 en su dev y 1599 en su test. Los ids son únicos.
- **Los 2885 tuits de la competencia están todos en HatEval:**
  - nuestro train: 1558 del train de HatEval y 173 de su dev;
  - nuestro test: 1041 del train y 113 del dev;
  - ninguno viene del test de HatEval.
- **Quedan 3714 tuits con un id que no tenemos:** 1901 del train de HatEval, 214 del dev y los 1599 de su test. Es el máximo que se podría añadir antes de buscar casi duplicados.

## 3. El mismo borrado de caracteres no ASCII (D3)

El train de la competencia perdió todos los caracteres no ASCII (`cállate` → `cllate`, H4). A todo texto externo se le aplica **el mismo borrado** (`remove_non_ascii`), sin transliterar, para que tenga el mismo aspecto que nuestro train.

**Comprobación:** en los tuits con el mismo id, se compara el texto de HatEval (ya con el borrado) con el nuestro.

In [4]:
has_non_ascii = H["text"].map(lambda t: not t.isascii())
print(f"Tuits de HatEval (es) con algún carácter no ASCII: {has_non_ascii.sum()} de {len(H)} ({has_non_ascii.mean():.1%})")
H["text_raw"] = H["text"]
H["text"] = H["text"].map(pp.remove_non_ascii)
for _, r in H[has_non_ascii].head(2).iterrows():
    print(f"   original: {r['text_raw'][:100]!r}\n   borrado:  {r['text'][:100]!r}")
rows = []
for name, d in [("train", train), ("test", test)]:
    m = d.merge(H, on="id", suffixes=("", "_h"))
    same = m["text"].str.strip() == m["text_h"].str.strip()
    q = md.quoted_mask(m["text"])
    rows.append({"nuestro": name, "ids comunes": len(m), "texto idéntico": f"{same.mean():.1%}",
                 "distintos": int((~same).sum()), "de ellos, con comillas (H3)": int((~same & q).sum()),
                 "de ellos, '#¿NOMBRE?'": int((~same & (m["text"] == "#¿NOMBRE?")).sum()),
                 "misma clave normalizada": f"{(m['text'].map(ex._near_duplicate_key) == m['text_h'].map(ex._near_duplicate_key)).mean():.1%}"})
display(pd.DataFrame(rows).set_index("nuestro"))
m = train.merge(H, on="id", suffixes=("", "_h"))
ex_q = m[md.quoted_mask(m["text"]) & (m["text"].str.strip() != m["text_h"].str.strip())].iloc[0]
print(f"Ejemplo con comillas:\n   nuestro: {ex_q['text'][:90]!r}\n   HatEval: {ex_q['text_h'][:90]!r}")

Tuits de HatEval (es) con algún carácter no ASCII: 4891 de 6599 (74.1%)
   original: "Easyjet quiere duplicar el número de mujeres piloto' Verás tú para aparcar el avión.. http://t.co/46"
   borrado:  "Easyjet quiere duplicar el nmero de mujeres piloto' Vers t para aparcar el avin.. http://t.co/46NuLk"
   original: 'El gobierno debe crear un control estricto de inmigración en las zonas fronterizas con Colombia por '
   borrado:  'El gobierno debe crear un control estricto de inmigracin en las zonas fronterizas con Colombia por q'


,ids comunes,texto idéntico,distintos,"de ellos, con comillas (H3)","de ellos, '#¿NOMBRE?'",misma clave normalizada
nuestro,,,,,,
train,1731,97.6%,42,39,0,99.8%
test,1154,97.1%,34,25,6,99.1%


Ejemplo con comillas:
   nuestro: '""T explicas el patriarcado como """"simios negratas"""" no vengas a exigir nada a las exp'
   HatEval: '"T explicas el patriarcado como ""simios negratas"" no vengas a exigir nada a las explicac'


**Interpretación:**
- **En HatEval, el 74.1% de los tuits tiene algún carácter no ASCII** (tildes, ñ, ¿, emojis). Con el borrado quedan como los nuestros: *número* → *nmero*, *avión* → *avin*.
- **Comparación con nuestros tuits del mismo id:** el texto es idéntico en el 97.6% del train y el 97.1% del test, y la clave normalizada coincide en el 99.8% y el 99.1%.
- **Las diferencias son de formato:**
  - 39 de las 42 del train y 25 de las 34 del test son tuits con comillas. En la competencia llevan **un nivel más de escape** (`""` → `""""`), que es el rastro del CSV que produjo el artefacto H3;
  - 6 son las celdas `#¿NOMBRE?` del test. HatEval tiene su texto original, pero ese texto es del test y **no se usa**.
- **Conclusión:** el borrado de D3 es la única transformación necesaria. Las comillas no se reescriben: la regla D34 ya se aplica después, sobre el texto de la competencia.

## 4. Deduplicación estricta (sin mirar etiquetas)

Cada tuit de HatEval se compara con **nuestro train y nuestro test**, en este orden; el primer criterio que se cumple decide:
1. **mismo id;**
2. **misma clave normalizada** (`_near_duplicate_key`: minúsculas, sin menciones, URLs, RT ni signos; la del 01);
3. **casi duplicado:** similitud coseno ≥ **0.85** entre las claves, con TF-IDF de n-gramas de caracteres (char_wb, 3–5).
   - El umbral pedido era 0.9, bajándolo si la revisión manual lo justificaba.
   - En la revisión (abajo), la franja 0.85–0.90 aún tiene duplicados reales: erratas (*"cllate vieja zorra"* / *"callate vieja zorra"*, 0.874) y letras alargadas. Por eso se fija **0.85**.
4. **duplicado interno:** misma clave que otro tuit de HatEval que ya se conserva.

**Coincidencias con el test:** se cuentan y se descartan. Si un tuit coincide a la vez con el train y con el test, cuenta como del test, que es lo más estricto.
**Coincidencias con el train:** se descartan también (son nuestros mismos tuits; dejarlos metería en el entrenamiento de un fold la etiqueta de un tuit de su validación). Antes se comparan sus etiquetas con las nuestras.

Las etiquetas se leen **después** del filtrado y solo para dos grupos: los tuits que se conservan y los que coinciden con el train. De los que coinciden con el test **no se leen**: el filtro de lectura de pyarrow ni siquiera los carga.

In [5]:
DEDUP = ex.deduplicate(H[["id", "text", "split"]], train, test, threshold=ex.COSINE_THRESHOLD)
ORDER = ["id-test", "clave-test", "coseno-test", "id-train", "clave-train", "coseno-train", "duplicado-interno", "se conserva"]
tab = pd.crosstab(DEDUP["status"], DEDUP["split"], margins=True, margins_name="total").reindex(ORDER + ["total"])
display(tab)
is_test = DEDUP["status"].str.endswith("-test")
is_train = DEDUP["status"].str.endswith("-train")
KEEP = DEDUP[DEDUP["status"] == "se conserva"].reset_index(drop=True)
print(f"Coinciden con nuestro TEST (descartados, sin leer etiquetas): {is_test.sum()} | con nuestro TRAIN: {is_train.sum()} | "
      f"duplicados internos: {(DEDUP['status'] == 'duplicado-interno').sum()} | se conservan: {len(KEEP)}")

after_key = DEDUP[DEDUP["status"].isin(["coseno-test", "coseno-train", "duplicado-interno", "se conserva"])]
sim = after_key[["sim_train", "sim_test"]].max(axis=1)
print("Sensibilidad al umbral (tuits que quedan tras id y clave):")
display(pd.DataFrame([{"umbral": t, "casi duplicados del train": int(((after_key["sim_train"] >= t) & (after_key["sim_test"] < t)).sum()),
                       "del test": int((after_key["sim_test"] >= t).sum())} for t in (0.95, 0.9, 0.85, 0.8, 0.75)]).set_index("umbral"))

split,dev,test,train,total
status,,,,
id-test,113,0,1041,1154
clave-test,0,44,0,44
coseno-test,0,16,2,18
id-train,173,0,1558,1731
clave-train,0,11,0,11
coseno-train,0,13,2,15
duplicado-interno,0,5,0,5
se conserva,214,1510,1897,3621
total,500,1599,4500,6599


Coinciden con nuestro TEST (descartados, sin leer etiquetas): 1216 | con nuestro TRAIN: 1757 | duplicados internos: 5 | se conservan: 3621
Sensibilidad al umbral (tuits que quedan tras id y clave):


,casi duplicados del train,del test
umbral,,
0.95,3,5
0.90,7,12
0.85,15,18
0.80,21,26
0.75,27,40


In [6]:
ours_text = pd.concat([train[["id", "text"]], test[["id", "text"]]]).set_index("id")["text"]
review = DEDUP[DEDUP["sim_train"].notna()].assign(sim=lambda d: d[["sim_train", "sim_test"]].max(axis=1))
for lo, hi, label in [(0.85, 0.90, "se descartan (0.85–0.90)"), (0.80, 0.85, "se conservan (0.80–0.85)")]:
    smp = review[(review["sim"] >= lo) & (review["sim"] < hi)].sort_values("sim", ascending=False).head(6)
    print(f"--- Revisión manual: {label}")
    for _, r in smp.iterrows():
        src = "test" if r["sim_test"] >= r["sim_train"] else "train"
        print(f"{r['sim']:.3f} [{src}] HatEval: {r['key'][:95]}")
        print(f"              nuestro: {ex._near_duplicate_key(pp.remove_non_ascii(ours_text[int(r[f'near_{src}'])]))[:95]}")

--- Revisión manual: se descartan (0.85–0.90)
0.896 [train] HatEval: callate y ponete a estudiar hijo de puta
              nuestro: callate y ponete a estudiar zorra
0.893 [train] HatEval: jajajajajajajajaja se lo mereca la perra
              nuestro: jajajajajajaja perra te lo merecas
0.885 [test] HatEval: cllate maldita puta
              nuestro: cllate maldita perra
0.875 [train] HatEval: cllate puta zorra borracha drogadictaaa
              nuestro: cllate puta zorra borracha drogadicta
0.874 [train] HatEval: cllate vieja zorra
              nuestro: callate vieja zorra
0.868 [train] HatEval: tu eres puta
              nuestro: y tu eres una puta
--- Revisión manual: se conservan (0.80–0.85)
0.849 [test] HatEval: que te calles puta
              nuestro: que te calles hijo de puta
0.847 [train] HatEval: te lo mereces hijo de perra
              nuestro: no te lo mereces perra
0.841 [test] HatEval: cllate vos puta de mierda
              nuestro: callate vos puta de mierda
0.840 

**Etiquetas.** Ahora se leen las de los tuits que se conservan y las de los que coinciden con el train, comprobando antes que ninguno coincide con el test.

In [7]:
TRAIN_MATCH = DEDUP[is_train].reset_index(drop=True)
assert not set(KEEP["id"]) & set(DEDUP.loc[is_test, "id"]) and not set(TRAIN_MATCH["id"]) & set(DEDUP.loc[is_test, "id"])
L_KEEP = ex.read_labels(KEEP["id"])
L_TM = ex.read_labels(TRAIN_MATCH["id"])
assert (L_KEEP["id"].to_numpy() == KEEP["id"].to_numpy()).all() and (L_TM["id"].to_numpy() == TRAIN_MATCH["id"].to_numpy()).all()
X_EXT, Y_EXT, EXT_IDS = KEEP["text"], L_KEEP[LABELS].to_numpy().astype(int), KEEP["id"].to_numpy()
print(f"Etiquetas leídas: {len(L_KEEP)} tuits conservados + {len(L_TM)} coincidentes con el train (0 del test)")

ours = train.set_index("id")
tm = TRAIN_MATCH.assign(**{f"{lab}_h": L_TM[lab].to_numpy() for lab in LABELS}, match_id=TRAIN_MATCH["match_id"].astype(int))
tm = tm.join(ours[LABELS], on="match_id")
tm["tipo"] = tm["status"].str.replace("-train", "")
rows = []
for tipo, g in [("todos", tm)] + list(tm.groupby("tipo")):
    rows.append({"coincidencia": tipo, "tuits": len(g), **{f"acuerdo {lab}": f"{(g[lab] == g[lab + '_h']).mean():.4f}" for lab in LABELS},
                 "las 3 iguales": f"{(g[LABELS].to_numpy() == g[[l + '_h' for l in LABELS]].to_numpy()).all(axis=1).mean():.4f}"})
display(pd.DataFrame(rows).set_index("coincidencia"))
print("AG: nuestra etiqueta (filas) frente a la de HatEval (columnas), coincidencias por id:")
by_id = tm[tm["tipo"] == "id"]
display(pd.crosstab(by_id["AG"], by_id["AG_h"]))
q = md.quoted_mask(ours.loc[by_id["match_id"], "text"])
dis_ag = by_id["AG"] != by_id["AG_h"]
print(f"Filas del artefacto de comillas (H3): {q.sum()} | su AG en HatEval: {by_id.loc[q, 'AG_h'].value_counts().sort_index().to_dict()} | "
      f"su HS en HatEval: {by_id.loc[q, 'HS_h'].value_counts().sort_index().to_dict()}")
print(f"Desacuerdos en AG: {dis_ag.sum()}, de ellos con comillas: {(dis_ag & q).sum()}")

Etiquetas leídas: 3621 tuits conservados + 1757 coincidentes con el train (0 del test)


,tuits,acuerdo HS,acuerdo TR,acuerdo AG,las 3 iguales
coincidencia,,,,,
todos,1757,0.9960,0.9960,0.9778,0.9778
clave,11,1.0000,1.0000,0.9091,0.9091
coseno,15,0.6667,0.6667,0.6667,0.6667
id,1731,0.9988,0.9988,0.9809,0.9809


AG: nuestra etiqueta (filas) frente a la de HatEval (columnas), coincidencias por id:


AG_h,0,1
AG,,
0,1069,0
1,33,629


Filas del artefacto de comillas (H3): 51 | su AG en HatEval: {0: 31, 1: 20} | su HS en HatEval: {0: 26, 1: 25}
Desacuerdos en AG: 33, de ellos con comillas: 31


**Interpretación:**
- **De 6599 tuits, se conservan 3621:**
  - **coinciden con nuestro test 1216, que se descartan sin leer sus etiquetas:** 1154 por id, 44 por clave normalizada y 18 por coseno;
  - coinciden con nuestro train 1757 (1731 / 11 / 15);
  - hay 5 duplicados internos.
- **Los conservados son 1897 del train de HatEval, 214 de su dev y 1510 de su test:** ≈2.6 veces nuestro fold de entrenamiento (1385).
- **Por clave hay más coincidencias con el test (44) que con el train (11).** Son plantillas cortas que se repiten (*"cállate puta…"*) y que también tienen gemelo en el train: ante train y test a la vez, gana el test.
- **Umbral del coseno:**
  - con 0.9 se quitarían 7 + 12 tuits; con 0.85, 15 + 18;
  - en la franja 0.85–0.90 aparecen duplicados reales: *"cllate vieja zorra"* / *"callate vieja zorra"* (0.874), *"drogadictaaa"* / *"drogadicta"* (0.875);
  - en 0.80–0.85 ya son tuits distintos con el mismo insulto (*"que te calles puta"* / *"que te calles hijo de puta"*), aunque alguno roza el límite (*"cllate vos puta de mierda"* / *"callate vos…"*, 0.841, del test).
  - **0.85 es razonable.** El control de fuga del §10 cubre lo que pueda quedar.
- **Etiquetas de las coincidencias con el train:**
  - por id: HS 0.9988, TR 0.9988, AG 0.9809;
  - **los 33 desacuerdos en AG son todos "nuestro AG=1, HatEval AG=0"**, y 31 de ellos tienen comillas;
  - las 51 filas del artefacto tienen en HatEval AG {0: 31, 1: 20} y HS {0: 26, 1: 25}.
  - **Confirmado: el AG=1 de esas filas lo puso el archivo de la competencia, no los anotadores.** Fuera del artefacto, las etiquetas son las mismas, así que HatEval está anotado igual que nuestros datos.
- **Las coincidencias por coseno con el train** tienen etiquetas distintas en 1 de cada 3. Son tuits parecidos pero no iguales, y descartarlos es conservador.

## 5. Compatibilidad: distribución de etiquetas y jerarquía

Antes de entrenar, se comprueba que los tuits conservados se parecen a los nuestros:
- **% de positivos y combinaciones** de etiquetas;
- **la jerarquía H2:** en nuestro train TR=1 ⇒ HS=1 siempre, y AG=1 ⇒ HS=1 salvo las 26 filas del artefacto;
- **el artefacto de comillas:** en nuestro train, el 100% de las filas con `;` o `"` tienen AG=1. Si en HatEval no pasa, se confirma que es un efecto del archivo de la competencia y no del lenguaje.
- **cómo se eligieron nuestros tuits:** qué fracción de HatEval (train + dev) acabó en la competencia, según la longitud del tuit.

In [8]:
def describe(Ymat, txt, name):
    combos = pd.Series(["".join(map(str, r)) for r in Ymat]).value_counts(normalize=True)
    q = md.quoted_mask(txt)
    return {"conjunto": name, "tuits": len(Ymat), **{f"% {lab}": round(100 * Ymat[:, j].mean(), 1) for j, lab in enumerate(LABELS)},
            "TR=1 ⇒ HS=1": f"{(Ymat[Ymat[:, 1] == 1, 0] == 1).mean():.3f}", "AG=1 ⇒ HS=1": f"{(Ymat[Ymat[:, 2] == 1, 0] == 1).mean():.3f}",
            **{f"% {c}": round(100 * combos.get(c, 0), 1) for c in ("000", "100", "101", "110", "111", "001")},
            "con comillas": int(q.sum()), "% AG=1 con comillas": round(100 * Ymat[q, 2].mean(), 1) if q.any() else np.nan,
            "longitud media": round(pd.Series(txt).str.len().mean(), 1)}


rows = [describe(Y, texts, "nuestro train")]
rows += [describe(Y_EXT[(KEEP["split"] == s).to_numpy()], X_EXT[(KEEP["split"] == s).to_numpy()], f"HatEval {s} (conservados)") for s in ex.SPLITS]
rows += [describe(Y_EXT, X_EXT, "HatEval conservados (todos)")]
display(pd.DataFrame(rows).set_index("conjunto").T)
print("Columna 'target' de HatEval (conservados):", L_KEEP["target"].value_counts().to_dict())
# ¿Cómo se eligieron nuestros tuits dentro de HatEval? Fracción de HatEval train+dev que está en la competencia, por longitud
hd = DEDUP[DEDUP["split"] != "test"].assign(longitud=lambda d: d["text"].str.len())
bins = pd.cut(hd["longitud"], [0, 60, 100, 140, 200, 1000], labels=["≤60", "61–100", "101–140", "141–200", ">200"])
sel = pd.DataFrame({"tuits HatEval (train+dev)": hd.groupby(bins, observed=False).size(),
                    "% en nuestro train o test": (100 * hd["status"].str.startswith("id-").groupby(bins, observed=False).mean()).round(1),
                    "conservados aquí": (hd["status"] == "se conserva").groupby(bins, observed=False).sum()})
display(sel.T)
print("Longitud (caracteres), cuartiles 25/50/75: nuestro train", texts.str.len().quantile([.25, .5, .75]).round().tolist(),
      "| HatEval conservados", X_EXT.str.len().quantile([.25, .5, .75]).round().tolist())

conjunto,nuestro train,HatEval train (conservados),HatEval dev (conservados),HatEval test (conservados),HatEval conservados (todos)
tuits,1731,1897,214,1510,3621
% HS,43.7,39.7,45.8,38.5,39.5
% TR,30.0,18.6,25.7,23.1,20.9
% AG,38.2,30.4,37.9,26.3,29.1
TR=1 ⇒ HS=1,1.000,1.000,1.000,1.000,1.000
AG=1 ⇒ HS=1,0.961,1.000,1.000,1.000,1.000
% 000,54.8,60.3,54.2,61.5,60.5
% 100,5.3,7.7,5.6,8.8,8.1
% 101,8.4,13.4,14.5,6.6,10.6
% 110,1.6,1.5,2.3,3.4,2.3


Columna 'target' de HatEval (conservados): {'mis': 2008, 'mig': 1613}


longitud,≤60,61–100,101–140,141–200,>200
tuits HatEval (train+dev),901.0,1278.0,1205.0,748.0,868.0
% en nuestro train o test,83.7,67.5,57.7,46.3,26.2
conservados aquí,145.0,413.0,510.0,402.0,641.0


Longitud (caracteres), cuartiles 25/50/75: nuestro train [59.0, 91.0, 131.0] | HatEval conservados [89.0, 135.0, 212.0]


**Interpretación:**
- **Mismo esquema de anotación:**
  - TR=1 ⇒ HS=1 y AG=1 ⇒ HS=1 se cumplen al 100% en HatEval. En nuestro train, AG=1 ⇒ HS=1 baja a 0.961 solo por las 26 filas 001 del artefacto, una combinación que en HatEval no existe;
  - **en HatEval, las filas con comillas tienen AG=1 en el 25.8%**, frente al 100% de las nuestras. Confirma de nuevo que el artefacto es del archivo de la competencia.
- **Distribución algo distinta:**
  - los conservados tienen menos positivos: HS 39.5%, TR 20.9%, AG 29.1%, frente a 43.7%, 30.0% y 38.2%;
  - tienen menos 111 (18.5% frente a 28.4%) y más 101 (10.6% frente a 8.4%);
  - el test de HatEval es el más alejado: AG 26.3%, 101 solo 6.6%;
  - hay más tuits contra inmigrantes: `target` mig 45%, frente a ≈34% en nuestro train.
- **La competencia eligió sobre todo tuits cortos:** está el 84% de los tuits de HatEval train+dev de ≤ 60 caracteres y solo el 26% de los de más de 200. Por eso los conservados son más largos: mediana 135 caracteres frente a 91.
- **Consecuencias:**
  - HatEval no es exactamente "más datos del mismo tipo", sino del mismo esquema con otra mezcla de longitudes y de etiquetas. La ganancia puede quedar por debajo de la extrapolada en el 09;
  - el peso 0.5 cubre en parte esa diferencia;
  - los umbrales se eligen siempre con nuestras OOF, así que la diferencia de prevalencias no los desplaza.

## 6. Protocolo de CV con HatEval

**En cada fold k** de nuestros folds:
- **entrenamiento:** nuestras filas de los otros 4 folds **+ todos los tuits de HatEval conservados**;
- **validación:** solo nuestras filas del fold k. Las OOF son siempre de nuestras filas, así que el stacker, los umbrales y cualquier métrica se calculan exactamente igual que en el 07–10 y **son comparables** con FINAL_A_quote fold a fold.

**Peso de HatEval.** Cada fila de HatEval entra con un peso en la pérdida: **1.0** (igual que las nuestras) o **0.5** (la mitad). Con 0.5 se cubre el caso de que HatEval se parezca algo menos a nuestro train que nuestro propio train (por ejemplo, su split de test se recogió en otra época).

**Dentro de cada modelo, nada de HatEval se valida:**
- **regresión logística y SVM lineal:** el C se vuelve a elegir con la **CV anidada** del 04 (3 folds internos) en cada fold externo. Más datos suelen pedir otra regularización. Los folds internos se hacen con nuestras filas; HatEval entra en el entrenamiento de cada fold interno, y la puntuación interna es con nuestras filas;
- **configuración por etiqueta** (09): igual, con su rejilla;
- **redes:** la validación interna del *early stopping* (12%) sale solo de nuestras filas; HatEval siempre entrena. Las CNN llevan **3 semillas** (en el pool v2 sin HatEval llevaban 10);
- **representación** (vocabulario, IDF, diccionario de restauración): se ajusta, sin etiquetas, con todos los textos de entrenamiento del fold, incluido HatEval.

**Referencia:** FINAL_A_quote (pool D del 07, stacking con las 3 etiquetas, regla de comillas): 0.8060 en `folds.csv` y 0.8019 en `folds_seed2`.

**Regla del 07 para adoptar un candidato:** Δ ≥ +0.005 frente a la referencia y positiva en ≥ 4/5 folds, en los dos conjuntos de folds. La selección para el "mejor de 10" usa además los criterios del 10 (sección 9).

In [9]:
def in_worker(fn, *args):
    """Ejecuta fn en un proceso nuevo (D33: torch y xgboost/LightGBM no conviven en un mismo proceso)."""
    ex._fresh_workers()
    out = Parallel(n_jobs=2)([delayed(fn)(*args)])[0]
    ex._fresh_workers()
    return out


def crossfit_rule(ids, method, fs, rule=True):
    """Crossfit del 07 y, con rule, la regla de comillas en las decisiones fuera de fold (como en el 08 y el 10).
    Devuelve el resultado del crossfit y las métricas por fold."""
    P = np.stack([PD[fs][m] for m in ids], axis=1)
    comb = en.Single(0) if method == "single" else en.make_combiner(method, "platt")
    r = en.crossfit(comb, P, Y, FOLDSETS[fs], EMPTY_TR)
    pred = r["pred"].copy()
    if rule:
        pred[QUOTED, AG] = 1
    f = FOLDSETS[fs]
    return r, pd.DataFrame([en.metrics(Y[f == k], pred[f == k]) for k in sorted(np.unique(f))])


def delta(a, b):
    d = en.paired_delta(a["F1_macro"], b["F1_macro"])
    return d, f"{d['Δ']:+.4f} ± {d['Δ desv.']:.4f} ({d['folds a favor']}/5)"


GRIDS_04 = {  # rejillas de la CV anidada del 04 (sin cambios)
    "logreg": [{"C": c} for c in (0.3, 1.0, 3.0, 10.0, 30.0)] + [{"C": c, "l1_ratio": 0.5} for c in (1.0, 3.0, 10.0)]
              + [{"C": c, "l1_ratio": 1.0} for c in (3.0, 10.0)],
    "linsvm": [{"C": c} for c in (0.03, 0.1, 0.3, 1.0, 3.0)],
}
CORE = ["V2__logreg__nested", "V1__logreg__nested", "V3__linsvm__nested"]
CLASSIC = ["V3__et__word__k2000", "V3__xgb__word__k500", "V3__nbsvm", "V3__mnb__word"]
NETS = ["V1__nn_cnn_char__tune-d0.3-lr0.002", "V1__nn_cnn_word__w2v", "V3__nn_mlp__tune-d0.5-lr0.002"]
POOL_D = CORE + CLASSIC + NETS                      # el pool del 07 (FINAL_A)
POOL_C = CORE + NETS
S10 = {NETS[0]: NETS[0] + "__s10", NETS[1]: NETS[1] + "__s10"}
POOL_DS10 = [S10.get(m, m) for m in POOL_D]
NEW_09 = ["V2__labelwise", "V2__conditional", "V2__powerset", "V3__nn_cnn_char__tune-d0.3-lr0.002__s10"]
POOL_V2 = POOL_DS10 + NEW_09                        # el pool v2 del 09 (14 modelos)
CNN_V3 = "V3__nn_cnn_char__tune-d0.3-lr0.002"       # la CNN de caracteres V3, aquí con 3 semillas

oof_store, test_store = pd.read_parquet(md.OOF_PATH), pd.read_parquet(md.TEST_PRED_PATH)


def stored(store, mid, ids):
    return store[store["model_id"] == mid].set_index("id").loc[np.asarray(ids), P_COLS].to_numpy()


PD = {"folds": {m: stored(oof_store, m, train["id"]) for m in POOL_D},
      "folds_seed2": {m: np.load(CACHE_07 / f"{m}__folds_seed2.npy") for m in POOL_D}}
for m in list(S10.values()) + NEW_09:
    for fs in FOLDSETS:
        PD[fs][m] = np.load(CACHE_09 / f"{m}__{fs}.npy")
P_TEST = {m: stored(test_store, m, test["id"]) for m in POOL_D}
P_TEST.update({m: np.load(CACHE_10 / f"{m}__test.npy") for m in list(S10.values()) + NEW_09})

REF = {fs: crossfit_rule(POOL_D, "stack-all", fs)[1] for fs in FOLDSETS}
print("FINAL_A_quote (referencia):", {fs: round(pf["F1_macro"].mean(), 4) for fs, pf in REF.items()})

FINAL_A_quote (referencia): {'folds': np.float64(0.806), 'folds_seed2': np.float64(0.8019)}


## 7. Los 14 modelos del pool v2, reentrenados con HatEval

**Mismas configuraciones que en el pool v2 del 09**, con dos cambios:
- el C de las logísticas y de la SVM lineal (y la configuración por etiqueta) se vuelve a elegir con CV anidada;
- las CNN llevan 3 semillas.

Cada modelo se reentrena con **peso 1.0** (sufijo `__he10`) y **peso 0.5** (`__he05`), y se calculan sus OOF en `folds.csv` y en `folds_seed2`: 14 × 2 × 2 = 56 validaciones cruzadas de 5 folds.

**Orden** (prioridad pedida): primero los lineales y las CNN, después árboles y Naive Bayes, y al final la CNN V3. Si la ejecución se corta, lo más importante ya estará en caché.

**Hardware:** no hay GPU CUDA en esta máquina (09 §4), así que las redes se entrenan en CPU.

In [10]:
WEIGHTS = {"10": 1.0, "05": 0.5}


def he(m, tag):
    return f"{m.replace('__s10', '')}__he{tag}"


def net_params(mid, **changes):
    params = md.load_model(mid).get_params()
    for k in ("n_threads", "thresholds", "fixed_epochs", "log_path"):
        params.pop(k, None)
    return dict(params, **changes)


def spec_of(m):
    """Cómo reentrenar con HatEval el miembro m del pool v2 (kind, parámetros) para external.fit_augmented."""
    base = m.replace("__s10", "")
    if base in CORE:
        r = REG.loc[base]
        return "nested", dict(variant=r["variant"], classifier=r["classifier"], features=r["features"], grid=GRIDS_04[r["classifier"]])
    if base in CLASSIC:
        r = REG.loc[base]
        return "sk", dict(variant=r["variant"], classifier=r["classifier"], features=r["features"],
                          clf_params=json.loads(r["params"]) or None)
    if base in ("V2__labelwise", "V2__conditional", "V2__powerset"):
        return base.split("__")[1], {}
    if base == CNN_V3:
        return "net", net_params(NETS[0], variant="V3", n_seeds=3)
    return "net", net_params(base, n_seeds=3)


PRIORITY = CORE + ["V2__labelwise", "V2__conditional", "V2__powerset", "V3__nbsvm"] + NETS + \
           ["V3__et__word__k2000", "V3__xgb__word__k500", "V3__mnb__word", CNN_V3]
SPECS = {m: spec_of(m) for m in PRIORITY}
POOL_V2_HE = {tag: [he(m, tag) for m in POOL_V2] for tag in WEIGHTS}
POOL_D_HE = {tag: [he(m, tag) for m in POOL_D] for tag in WEIGHTS}
INFO, TIMES = {}, {}
n_fold = np.mean([(folds != k).sum() for k in np.unique(folds)])
print(f"Por fold: {n_fold:.0f} tuits nuestros + {len(X_EXT)} de HatEval = {(n_fold + len(X_EXT)) / n_fold:.2f}× los datos")


def run_he(m, tag, fs, spec=None, weight=None):
    """OOF de la versión con HatEval de m (caché por fold en CACHE_DIR/<nombre>__<fs>/)."""
    name = he(m, tag) if tag else m
    spec = spec or SPECS[m]
    start = time.perf_counter()
    res = ex.augmented_cv(spec, texts, Y, FOLDSETS[fs], EXT_IDS, X_EXT, Y_EXT, WEIGHTS.get(tag, weight or 1.0),
                          CACHE_DIR / f"{name}__{fs}", n_jobs=N_JOBS, n_threads=FOLD_THREADS)
    PD[fs][name], INFO[(name, fs)], TIMES[(name, fs)] = res["oof"], res["info"], res["fit_seconds"]
    if res["cached_folds"] < 5:
        print(f"    · {name} [{fs}] {time.perf_counter() - start:.0f} s", flush=True)
    return name

Por fold: 1385 tuits nuestros + 3621 de HatEval = 3.61× los datos


In [11]:
start = time.perf_counter()
for fs in ("folds", "folds_seed2"):
    for tag in WEIGHTS:
        for m in PRIORITY:
            run_he(m, tag, fs)
        print(f"{fs}, peso {WEIGHTS[tag]}: listo | {(time.perf_counter() - start) / 60:.0f} min acumulados", flush=True)
total = sum(TIMES.values())
print(f"Tiempo de entrenamiento (reloj, suma de las 56 CV, incluidas ejecuciones anteriores): {total / 3600:.1f} h")

folds, peso 1.0: listo | 0 min acumulados


folds, peso 0.5: listo | 0 min acumulados


folds_seed2, peso 1.0: listo | 0 min acumulados


folds_seed2, peso 0.5: listo | 0 min acumulados


Tiempo de entrenamiento (reloj, suma de las 56 CV, incluidas ejecuciones anteriores): 6.3 h


**Cada modelo por separado.** F1 macro con el protocolo del 07 (umbral de meseta elegido sin mirar el fold y jerarquía), **sin** la regla de comillas, que afecta igual a todos. Se compara cada modelo con su versión sin HatEval:
- las CNN de caracteres y de palabras, con su versión de 3 semillas (las del pool D);
- la CNN V3, con la de 10 semillas (la única que existe sin HatEval).

In [12]:
def single(mid, fs):
    return crossfit_rule([mid], "single", fs, rule=False)[1]


NO_HE = {m: (NETS[0] if m == S10[NETS[0]] else NETS[1] if m == S10[NETS[1]] else m) for m in POOL_V2}
rows = []
for m in POOL_V2:
    row = {"modelo": NO_HE[m]}
    for fs in FOLDSETS:
        base = single(NO_HE[m], fs)
        row[f"sin HatEval [{fs}]"] = round(base["F1_macro"].mean(), 4)
        for tag, w in WEIGHTS.items():
            d, s = delta(single(he(m, tag), fs), base)
            row[f"Δ peso {w} [{fs}]"] = s
            row[f"_d{tag}_{fs}"] = d["Δ"]
    if SPECS[m.replace("__s10", "")][0] == "nested":
        row["C elegido (peso 1.0, folds)"] = ", ".join(json.dumps(i["params"]) for i in INFO[(he(m, "10"), "folds")])
    rows.append(row)
SINGLE = pd.DataFrame(rows).set_index("modelo")
display(SINGLE[[c for c in SINGLE.columns if not c.startswith("_")]])
for tag, w in WEIGHTS.items():
    print(f"Peso {w}: el modelo mejora en {int((SINGLE[f'_d{tag}_folds'] > 0).sum())}/14 en folds.csv y "
          f"{int((SINGLE[f'_d{tag}_folds_seed2'] > 0).sum())}/14 en folds_seed2 | Δ medio "
          f"{SINGLE[f'_d{tag}_folds'].mean():+.4f} / {SINGLE[f'_d{tag}_folds_seed2'].mean():+.4f}")

,sin HatEval [folds],Δ peso 1.0 [folds],Δ peso 0.5 [folds],sin HatEval [folds_seed2],Δ peso 1.0 [folds_seed2],Δ peso 0.5 [folds_seed2],"C elegido (peso 1.0, folds)"
modelo,,,,,,,
V2__logreg__nested,0.7902,+0.0009 ± 0.0083 (3/5),+0.0139 ± 0.0157 (4/5),0.7852,+0.0168 ± 0.0093 (5/5),+0.0229 ± 0.0103 (5/5),"{""C"": 3.0, ""l1_ratio"": 0.5}, {""C"": 3.0, ""l1_ratio"": 0.5}, {""C"": 3.0, ""l1_ratio"": 0.5}, {""C"": 3.0}, {""C"": 3..."
V1__logreg__nested,0.7871,+0.0087 ± 0.0126 (3/5),+0.0168 ± 0.0164 (5/5),0.7930,+0.0092 ± 0.0053 (5/5),+0.0159 ± 0.0033 (5/5),"{""C"": 30.0}, {""C"": 3.0, ""l1_ratio"": 0.5}, {""C"": 3.0, ""l1_ratio"": 0.5}, {""C"": 3.0}, {""C"": 3.0, ""l1_ratio"": ..."
V3__linsvm__nested,0.7815,+0.0092 ± 0.0175 (4/5),+0.0190 ± 0.0111 (5/5),0.7818,+0.0154 ± 0.0115 (5/5),+0.0185 ± 0.0141 (5/5),"{""C"": 1.0}, {""C"": 1.0}, {""C"": 0.3}, {""C"": 0.3}, {""C"": 1.0}"
V3__et__word__k2000,0.7631,+0.0195 ± 0.0089 (5/5),+0.0224 ± 0.0095 (5/5),0.7519,+0.0324 ± 0.0121 (5/5),+0.0339 ± 0.0134 (5/5),NaN
V3__xgb__word__k500,0.7469,+0.0266 ± 0.0060 (5/5),+0.0216 ± 0.0101 (5/5),0.7471,+0.0309 ± 0.0100 (5/5),+0.0258 ± 0.0074 (5/5),NaN
V3__nbsvm,0.7702,+0.0215 ± 0.0079 (5/5),+0.0198 ± 0.0054 (5/5),0.7701,+0.0228 ± 0.0235 (4/5),+0.0175 ± 0.0161 (4/5),NaN
V3__mnb__word,0.7346,+0.0065 ± 0.0212 (3/5),+0.0042 ± 0.0148 (3/5),0.7399,+0.0053 ± 0.0154 (4/5),+0.0025 ± 0.0143 (3/5),NaN
V1__nn_cnn_char__tune-d0.3-lr0.002,0.7621,+0.0148 ± 0.0092 (4/5),+0.0194 ± 0.0093 (5/5),0.7670,+0.0124 ± 0.0130 (4/5),+0.0209 ± 0.0141 (5/5),NaN
V1__nn_cnn_word__w2v,0.7635,+0.0418 ± 0.0223 (5/5),+0.0386 ± 0.0233 (5/5),0.7742,+0.0291 ± 0.0183 (5/5),+0.0317 ± 0.0172 (5/5),NaN


Peso 1.0: el modelo mejora en 14/14 en folds.csv y 14/14 en folds_seed2 | Δ medio +0.0159 / +0.0185
Peso 0.5: el modelo mejora en 14/14 en folds.csv y 14/14 en folds_seed2 | Δ medio +0.0183 / +0.0195


**Comparación con la curva de aprendizaje del 09** (H60). Allí se extrapoló la ganancia de pasar de ≈1385 tuits por fold a ≈3.1× (los 1731 tuits + ≈3.7k más):
- lineal (V2, logística elasticnet con C=10): **+0.033** con la curva de potencia y +0.051 con la logarítmica; rango razonable **+0.01 a +0.05**;
- CNN de caracteres: +0.042 y +0.056.

Aquí los datos por fold se multiplican por el factor impreso arriba (≈3.6×), algo más de lo extrapolado.

In [13]:
LC_09 = {"V2__logreg__nested": ("lineal V2", 0.033, 0.051), NETS[0]: ("CNN de caracteres", 0.042, 0.056)}
rows = []
for m, (label, pw, lg) in LC_09.items():
    for tag, w in WEIGHTS.items():
        rows.append({"modelo": label, "peso": w, "extrapolado 09 (potencia / log)": f"+{pw:.3f} / +{lg:.3f}",
                     **{f"observado [{fs}]": f"{SINGLE.loc[m, f'_d{tag}_{fs}']:+.4f}" for fs in FOLDSETS}})
display(pd.DataFrame(rows).set_index(["modelo", "peso"]))

extrapolado 09 (potencia / log) observado [folds] observado [folds_seed2]
modelo            peso                                                                          
lineal V2         1.0                  +0.033 / +0.051           +0.0009                 +0.0168
                  0.5                  +0.033 / +0.051           +0.0139                 +0.0229
CNN de caracteres 1.0                  +0.042 / +0.056           +0.0148                 +0.0124
                  0.5                  +0.042 / +0.056           +0.0194                 +0.0209

**Interpretación:**
- **HatEval mejora a los 14 modelos, con los dos pesos y en los dos conjuntos de folds** (14/14 en todos los casos). Δ medio:
  - peso 1.0: +0.0159 en `folds.csv` y +0.0185 en `folds_seed2`;
  - peso 0.5: +0.0183 y +0.0195.
- **Quién gana más:**
  - la CNN de palabras: +0.042 / +0.029 con peso 1.0. Es la que aprende sus embeddings desde cero con el texto del fold (H39), así que es la que más depende de la cantidad de texto;
  - el MLP (+0.027), xgboost (+0.027 / +0.031) y ExtraTrees (+0.020 / +0.032).
- **Quién gana menos:**
  - Naive Bayes multinomial: +0.007 / +0.005;
  - la logística V2 con peso 1.0: +0.0009 (3/5) en `folds.csv`, aunque +0.017 (5/5) en `folds_seed2`.
- **Con peso 0.5 ganan más los lineales:**
  - logística V2: +0.014 / +0.023, frente a +0.001 / +0.017 con peso 1.0;
  - logística V1: +0.017 / +0.016 frente a +0.009 / +0.009;
  - SVM lineal: +0.019 / +0.019 frente a +0.009 / +0.015.
  - Los árboles, xgboost, NB-SVM y el MLP van algo mejor con 1.0. Es coherente con el §5: HatEval no es exactamente nuestra distribución, y los lineales, que se ajustan más a ella, prefieren que pese menos.
- **La CV anidada elige otra regularización:** la logística V2 pasa del C=10 del 04 a C=3 (elasticnet) en casi todos los folds.
- **Frente a la curva de aprendizaje del 09:**
  - con 3.61× los datos por fold (algo más que el 3.1× extrapolado), el lineal gana +0.001 a +0.023 y la CNN de caracteres +0.012 a +0.021;
  - es **la parte baja del rango previsto** (+0.01 a +0.05) y menos que el valor central (+0.033 / +0.042);
  - lo que se esperaba si HatEval no es "más datos del mismo tipo" sino del mismo esquema con otra mezcla de longitudes y etiquetas (§5).
- **Coste:** 6.3 h de CV en total. Lo más lento fue la configuración por etiqueta con CV anidada (34–49 min por CV), seguida de las CNN de caracteres (≈14–16 min).

## 8. Candidatos con HatEval

Todos con el método de FINAL_A (stacking de las 3 etiquetas) salvo que se diga otra cosa, con la regla de comillas y evaluados en los dos conjuntos de folds, con la diferencia emparejada por fold frente a FINAL_A_quote:

| Candidato | Pool | Método |
|---|---|---|
| **H1** | pool v2 con HatEval, peso 1.0 (14 modelos) | stacking |
| **H2** | pool v2 con HatEval, peso 0.5 (14) | stacking |
| **H3** | pool D (los 10 del 07) con HatEval, con el mejor peso de H1/H2 | stacking |
| **H4** "mixto" | los 14 sin HatEval + los 14 con HatEval (mejor peso): 28 modelos | stacking |
| **H5** | el mejor de H1–H3 | pesos ≥ 0 por log-loss (el *weights* del 07) |
| **H6** | el mejor de H1–H3 + las redes nuevas que se admitan (8.2) | stacking |

**El "mejor peso" y "el mejor de H1–H3"** se eligen por la CV media de los dos conjuntos de folds.

**Fold-bag (opcional): no se genera.** En el 10, A_foldbag_quote tenía por construcción la misma CV que FINAL_A_quote y quedó 0.0014 por debajo en el leaderboard público. Aquí costaría guardar y volver a predecir 70 modelos de fold por candidato.

### 8.1 H1 y H2: ¿qué peso para HatEval?

In [14]:
CANDS, CV = {}, {}


def evaluate(tag, ids, method):
    CANDS[tag] = (ids, method)
    CV[tag] = {fs: crossfit_rule(ids, method, fs) for fs in FOLDSETS}


def cv_row(tag):
    row = {"candidato": tag, "modelos": len(CANDS[tag][0]), "método": CANDS[tag][1]}
    ok = True
    for fs in FOLDSETS:
        pf = CV[tag][fs][1]
        d, s = delta(pf, REF[fs])
        row[f"F1 [{fs}]"] = round(pf["F1_macro"].mean(), 4)
        row[f"Δ vs FINAL_A_quote [{fs}]"] = s
        ok &= d["Δ"] >= 0.005 and d["folds a favor"] >= 4
    row["CV media"] = round(np.mean([CV[tag][fs][1]["F1_macro"].mean() for fs in FOLDSETS]), 4)
    row["cumple la regla del 07"] = ok
    return row


evaluate("H1_quote", POOL_V2_HE["10"], "stack-all")
evaluate("H2_quote", POOL_V2_HE["05"], "stack-all")
display(pd.DataFrame([cv_row(t) for t in ("H1_quote", "H2_quote")]).set_index("candidato"))
BEST_TAG = max(WEIGHTS, key=lambda t: cv_row(f"H{1 if t == '10' else 2}_quote")["CV media"])
print(f"Mejor peso: {WEIGHTS[BEST_TAG]} (sufijo __he{BEST_TAG})")

,modelos,método,F1 [folds],Δ vs FINAL_A_quote [folds],F1 [folds_seed2],Δ vs FINAL_A_quote [folds_seed2],CV media,cumple la regla del 07
candidato,,,,,,,,
H1_quote,14,stack-all,0.8219,+0.0158 ± 0.0116 (5/5),0.8251,+0.0232 ± 0.0144 (5/5),0.8235,True
H2_quote,14,stack-all,0.8201,+0.0140 ± 0.0148 (4/5),0.8283,+0.0263 ± 0.0121 (5/5),0.8242,True


Mejor peso: 0.5 (sufijo __he05)


### 8.2 Miembros nuevos

Dos redes que solo tienen sentido con más datos (fijadas antes de mirar resultados):
- **CNN de palabras con Word2Vec "ampliado"** (`V1__nn_cnn_word__w2v-hetext`). En H1/H2, la CNN de palabras ya entrena su Word2Vec con el texto del fold + HatEval, porque Word2Vec usa todos los textos de entrenamiento. Este miembro aísla **solo el texto**: Word2Vec con el texto del fold + HatEval, pero la red aprende **solo con nuestras etiquetas**. Así mide cuánto aporta HatEval sin sus etiquetas, y da un miembro distinto;
- **el mejor Transformer del 06** (`V1__nn_trf_bpe__v800-d128-l2-aug`, BPE de 800 tokens con aumento de datos; 0.7052 sin HatEval), reentrenado con HatEval y el mejor peso. Los Transformer desde cero eran los que más sufrían por falta de datos (H45, H47).

**Criterio de admisión** (el del 09): a menos de 0.04 del mejor lineal con HatEval **o** correlación de errores < 0.65 con él, en `folds.csv`. Solo los admitidos se calculan también en `folds_seed2`.

In [15]:
TRF = "V1__nn_trf_bpe__v800-d128-l2-aug"
NEW_SPECS = {"V1__nn_cnn_word__w2v-hetext": (None, ("net-text", net_params(NETS[1], n_seeds=3))),
             he(TRF, BEST_TAG): (BEST_TAG, ("trf", net_params("V1__nn_trf_bpe__v800-d128-l2", augment=True)))}
# el 06 no guardó el modelo con aumento: se parte del mismo sin aumento (guardado) y se comprueba que su configuración es la registrada
trf_reg = json.loads(REG.loc[TRF, "params"])
assert all(NEW_SPECS[he(TRF, BEST_TAG)][1][1][k] == v for k, v in trf_reg.items()), "configuración del Transformer distinta de la del 06"
best_lin = max((he(m, BEST_TAG) for m in CORE), key=lambda m: single(m, "folds")["F1_macro"].mean())
r_lin = crossfit_rule([best_lin], "single", "folds", rule=False)[0]
rows, ADMITTED_NEW = [], []
for name, (tag, spec) in NEW_SPECS.items():
    run_he(name, None, "folds", spec=spec, weight=WEIGHTS[tag] if tag else 1.0)
    r = crossfit_rule([name], "single", "folds", rule=False)
    gap = r_lin["folds"][("plateau", True)]["F1_macro"].mean() - r[1]["F1_macro"].mean()
    corr = np.corrcoef((r[0]["pred"] != Y).ravel(), (r_lin["pred"] != Y).ravel())[0, 1]
    ok = gap < 0.04 or corr < 0.65
    ADMITTED_NEW += [name] if ok else []
    rows.append({"miembro": name, "F1 macro": round(r[1]["F1_macro"].mean(), 4), "distancia al mejor lineal": round(gap, 4),
                 "correlación de errores": round(corr, 3), "entra": ok, "minutos de CV": round(TIMES[(name, "folds")] / 60, 1)})
display(pd.DataFrame(rows).set_index("miembro"))
print(f"Mejor lineal con HatEval: {best_lin} | admitidos: {ADMITTED_NEW}")
for name in ADMITTED_NEW:
    tag, spec = NEW_SPECS[name]
    run_he(name, None, "folds_seed2", spec=spec, weight=WEIGHTS[tag] if tag else 1.0)

,F1 macro,distancia al mejor lineal,correlación de errores,entra,minutos de CV
miembro,,,,,
V1__nn_cnn_word__w2v-hetext,0.7660,0.0381,0.602,True,2.2
V1__nn_trf_bpe__v800-d128-l2-aug__he05,0.7557,0.0485,0.547,True,20.5


Mejor lineal con HatEval: V2__logreg__nested__he05 | admitidos: ['V1__nn_cnn_word__w2v-hetext', 'V1__nn_trf_bpe__v800-d128-l2-aug__he05']


### 8.3 H3–H6

In [16]:
evaluate("H3_quote", POOL_D_HE[BEST_TAG], "stack-all")
evaluate("H4_quote", POOL_V2 + POOL_V2_HE[BEST_TAG], "stack-all")
BEST_H = max(["H1_quote", "H2_quote", "H3_quote"], key=lambda t: cv_row(t)["CV media"])
evaluate("H5_quote", CANDS[BEST_H][0], "weights")
if ADMITTED_NEW:
    evaluate("H6_quote", CANDS[BEST_H][0] + ADMITTED_NEW, "stack-all")
H_TAGS = list(CANDS)
H_TABLE = pd.DataFrame([cv_row(t) for t in H_TAGS]).set_index("candidato")
display(H_TABLE)
print(f"Mejor de H1–H3: {BEST_H}" + ("" if ADMITTED_NEW else " | H6 no se genera: ninguna red nueva se admitió"))

,modelos,método,F1 [folds],Δ vs FINAL_A_quote [folds],F1 [folds_seed2],Δ vs FINAL_A_quote [folds_seed2],CV media,cumple la regla del 07
candidato,,,,,,,,
H1_quote,14,stack-all,0.8219,+0.0158 ± 0.0116 (5/5),0.8251,+0.0232 ± 0.0144 (5/5),0.8235,True
H2_quote,14,stack-all,0.8201,+0.0140 ± 0.0148 (4/5),0.8283,+0.0263 ± 0.0121 (5/5),0.8242,True
H3_quote,10,stack-all,0.8193,+0.0133 ± 0.0156 (4/5),0.8286,+0.0266 ± 0.0124 (5/5),0.8240,True
H4_quote,28,stack-all,0.8238,+0.0177 ± 0.0101 (5/5),0.8233,+0.0214 ± 0.0112 (5/5),0.8235,True
H5_quote,14,weights,0.8216,+0.0156 ± 0.0109 (5/5),0.8275,+0.0256 ± 0.0125 (5/5),0.8246,True
H6_quote,16,stack-all,0.8202,+0.0142 ± 0.0112 (5/5),0.8272,+0.0253 ± 0.0099 (5/5),0.8237,True


Mejor de H1–H3: H2_quote


**Interpretación:**
- **H1 y H2: el peso apenas importa en el ensamble.**
  - H1 (peso 1.0): 0.8219 / 0.8251, con Δ +0.0158 (5/5) / +0.0232 (5/5);
  - H2 (peso 0.5): 0.8201 / 0.8283, con Δ +0.0140 (4/5) / +0.0263 (5/5).
  - Por la regla fijada (CV media) gana el **peso 0.5**: 0.8242 frente a 0.8235. Una diferencia de 0.0007 es ruido.
- **Miembros nuevos: los dos se admiten por correlación de errores** (< 0.65), no por calidad.
  - **CNN de palabras con Word2Vec ampliado** (solo el texto de HatEval): 0.7660, frente a 0.7635 sin HatEval. El texto solo aporta ≈+0.003; casi todo el +0.04 de la CNN de palabras viene de las **etiquetas** de HatEval.
  - **Transformer del 06 con HatEval:** de 0.7052 a **0.7557 (+0.050)**, la mayor mejora individual del notebook. Encaja con H45/H47: era el modelo más hambriento de datos. Aun así queda 0.049 por debajo del mejor lineal.
- **H3–H6 (mejor peso 0.5):**
  - H3, el pool D sin los 4 miembros del 09: 0.8193 / 0.8286. Esos 4 modelos siguen sin aportar en el stacking (como en el 09, H65);
  - H4 "mixto" (28 modelos): el mejor en `folds.csv` (0.8238), pero el peor en `folds_seed2` (0.8233);
  - H5 (pesos ≥ 0): 0.8216 / 0.8275, **la mejor CV media (0.8246)**;
  - H6 (H2 + las 2 redes nuevas): 0.8202 / 0.8272. No mejora a H2: el stacker ya tiene redes de sobra.
- **Los 6 H cumplen la regla del 07** (Δ ≥ +0.005 y ≥ 4/5 folds en los dos conjuntos). Es la primera vez desde el 07 que un candidato la cumple.
  - Ganancia frente a FINAL_A_quote: **+0.013 a +0.018 en `folds.csv` y +0.021 a +0.027 en `folds_seed2`**, unas 2 veces lo que aportó el ensamble (+0.011).
  - Entre los H, las diferencias de CV media son ≤ 0.001. Su orden no es significativo.
- **Envíos:** los 6 archivos pasan todas las comprobaciones:
  - 1154 filas, ids en orden, valores 0/1 y sin nulos;
  - coinciden con las probabilidades guardadas;
  - recarga de disco = memoria (diferencia 0.0).
  - El aviso de joblib *"A worker stopped while some jobs were given to the executor"* es el cierre de procesos inactivos de loky. No afecta a los resultados (las comprobaciones lo confirman).

### 8.4 Modelos finales y envíos

**Modelos finales de los miembros:** cada miembro con HatEval que use algún candidato se reentrena con **nuestros 1731 tuits + los de HatEval** y se guarda en `models/11_hateval/`. Se sigue lo mismo que en la CV:
- C elegido con la CV interna sobre nuestras filas;
- redes con épocas fijas = la mediana de las mejores épocas de sus folds en `folds.csv`.

Se entrenan en tres tandas de procesos separados (D33): scikit-learn, xgboost y las redes. Los miembros sin HatEval (H4) son los modelos finales ya guardados en el 07–10.

**Ensamble final de cada candidato** (como en el 10):
- el combinador y los umbrales se ajustan con las OOF de `folds.csv`;
- `EnsembleClassifier` con `quote_rule=True`, guardado en `models/11_hateval/`;
- el CSV se escribe con `make_submission`, que vuelve a cargar los modelos de disco y predice el test desde el texto crudo.

**Comprobaciones:** 1154 filas, ids en orden, valores 0/1, sin nulos; el archivo coincide con las probabilidades guardadas; y al recargar el ensamble de disco se obtiene lo mismo que en memoria.

In [17]:
def _final_job(name, spec, weight, epochs):
    import external as ex_  # el proceso nuevo importa torch solo si es una red (D33)

    return ex_.fit_final_augmented(spec, name, texts, Y, X_EXT, Y_EXT, weight, test["text"],
                                   {"origin": "11 (pool v2 + HatEval)"}, notebook="11", n_threads=FINAL_THREADS,
                                   fixed_epochs=epochs)


NEEDED = sorted({m for t in H_TAGS for m in CANDS[t][0] if m not in P_TEST})
JOBS = {}
for name in NEEDED:
    if name in NEW_SPECS:
        tag, spec = NEW_SPECS[name]
        weight = WEIGHTS[tag] if tag else 1.0
    else:
        base, tag = name.rsplit("__he", 1)
        spec, weight = SPECS[base], WEIGHTS[tag]
    epochs = ex.final_epochs(INFO[(name, "folds")]) if spec[0] in ("net", "net-text", "trf") else None
    JOBS[name] = (spec, weight, epochs)
def batch_of(spec):
    """Tanda de procesos (D33): xgboost/LightGBM, el resto de scikit-learn y torch traen cada uno su runtime de
    OpenMP y no pueden compartir proceso; cada tanda empieza con procesos nuevos."""
    if spec[0] in ("net", "net-text", "trf"):
        return "redes"
    return "xgboost" if spec[1].get("classifier") in ("xgb", "lgbm") else "scikit-learn"


for batch in ("scikit-learn", "xgboost", "redes"):
    todo = [n for n, (spec, _, _) in JOBS.items() if batch_of(spec) == batch
            and not ((CACHE_DIR / f"{n}__test.npy").exists() and md.find_model_path(n) is not None)]
    if todo:
        start = time.perf_counter()
        print(f"  finales ({batch}): {len(todo)} modelos ...", end=" ", flush=True)
        ex._fresh_workers()
        out = Parallel(n_jobs=FINAL_JOBS)(delayed(_final_job)(n, *JOBS[n]) for n in todo)
        ex._fresh_workers()
        for n, p in zip(todo, out):
            np.save(CACHE_DIR / f"{n}__test.npy", p)
        print(f"{(time.perf_counter() - start) / 60:.1f} min", flush=True)
for name in NEEDED:
    P_TEST[name] = np.load(CACHE_DIR / f"{name}__test.npy")
print(f"Modelos finales con HatEval: {len(NEEDED)} | probabilidades de test disponibles: {len(P_TEST)}")

Modelos finales con HatEval: 30 | probabilidades de test disponibles: 46


In [18]:
SLUG = {"H1_quote": "v2he10", "H2_quote": "v2he05", "H3_quote": f"Dhe{BEST_TAG}", "H4_quote": f"mix{BEST_TAG}"}
SLUG.update(H5_quote=SLUG[BEST_H], H6_quote=SLUG[BEST_H] + "-new")
FILES, MODELS_11, rows = {}, {}, []
for tag in H_TAGS:
    ids, method = CANDS[tag]
    name = f"ens11__{SLUG[tag]}__{method}__quote"
    r = CV[tag]["folds"][0]
    P_main = np.stack([PD["folds"][m] for m in ids], axis=1)
    fitted, thr = en.fit_final(en.make_combiner(method, "platt"), P_main, Y, folds, r["proba"])
    model = en.EnsembleClassifier(ids, fitted, thr, name=name, quote_rule=True)
    cv_info = {fs: round(CV[tag][fs][1]["F1_macro"].mean(), 4) for fs in FOLDSETS}
    en.save_ensemble(model, name, {"notebook": "11", "role": tag, "pool": ids, "method": method, "cv_F1_macro_with_rule": cv_info,
                                   "external": {"corpus": ex.HATEVAL_REPO, "revision": ex.HATEVAL_REVISION, "rows": len(X_EXT),
                                                "license": ex.HATEVAL_LICENSE}, "seed": SEED})
    sub_path = md.make_submission(name, filename=f"{tag}_{name}")
    sub = pd.read_csv(sub_path)
    P_te = np.stack([P_TEST[m] for m in ids], axis=1)
    expected = md.decide(model.predict_proba_from_base(P_te), test["text"], thr)
    expected[QUOTED_TE, AG] = 1
    p_disk = md.load_model(name).predict_proba_from_base(P_te)
    rows.append({"candidato": tag, "archivo": sub_path.name, "filas": len(sub), "ids en orden": bool((sub["id"] == test["id"]).all()),
                 "valores": sorted(np.unique(sub[LABELS]).tolist()), "nulos": int(sub.isna().sum().sum()),
                 "= probabilidades guardadas": bool((sub[LABELS].to_numpy() == expected).all()),
                 "recarga |disco − memoria|": float(np.abs(p_disk - model.predict_proba_from_base(P_te)).max()),
                 **{f"% {lab}=1": round(100 * sub[lab].mean(), 1) for lab in LABELS}, "umbrales": thr})
    FILES[tag] = sub_path.name
    MODELS_11[tag] = {"name": name, "model": model, "P_te": P_te, "oof": r["proba"], "ids": ids}
checks = pd.DataFrame(rows).set_index("candidato")
display(checks)
assert checks["= probabilidades guardadas"].all() and (checks["recarga |disco − memoria|"] == 0).all()
assert (checks["filas"] == len(test)).all() and checks["ids en orden"].all() and (checks["nulos"] == 0).all()

,archivo,filas,ids en orden,valores,nulos,= probabilidades guardadas,recarga |disco − memoria|,% HS=1,% TR=1,% AG=1,umbrales
candidato,,,,,,,,,,,
H1_quote,H1_quote_ens11__v2he10__stack-all__quote.csv,1154,True,"[0, 1]",0,True,0.0,45.4,30.8,39.4,"{'HS': 0.38, 'TR': 0.37, 'AG': 0.41}"
H2_quote,H2_quote_ens11__v2he05__stack-all__quote.csv,1154,True,"[0, 1]",0,True,0.0,44.2,30.4,38.9,"{'HS': 0.39, 'TR': 0.37, 'AG': 0.39}"
H3_quote,H3_quote_ens11__Dhe05__stack-all__quote.csv,1154,True,"[0, 1]",0,True,0.0,44.0,30.2,38.1,"{'HS': 0.37, 'TR': 0.34, 'AG': 0.39}"
H4_quote,H4_quote_ens11__mix05__stack-all__quote.csv,1154,True,"[0, 1]",0,True,0.0,47.0,30.6,41.3,"{'HS': 0.37, 'TR': 0.37, 'AG': 0.39}"
H5_quote,H5_quote_ens11__v2he05__weights__quote.csv,1154,True,"[0, 1]",0,True,0.0,45.8,30.4,40.1,"{'HS': 0.37, 'TR': 0.36, 'AG': 0.37}"
H6_quote,H6_quote_ens11__v2he05-new__stack-all__quote.csv,1154,True,"[0, 1]",0,True,0.0,43.7,30.5,38.0,"{'HS': 0.39, 'TR': 0.39, 'AG': 0.41}"


## 9. Selección para el "mejor de 10"

Se aplican los **criterios del 10** (D40), fijados de antemano, a los candidatos del 10 y a los H:
- **a) Calidad:** CV con la regla a ≤ 0.010 del mejor, en los dos conjuntos de folds.
- **b) Diversidad:** ≥ 1% de las 3462 celdas del test distintas de cada candidato admitido con mejor CV.
- **c) Orden** por la CV media. El leaderboard público solo **descarta** (más de 0.010 por debajo del mejor puntaje público), nunca ordena.
- **d)** FINAL_A (sin regla) solo como cobertura si sobra hueco; `probe_HS_only` nunca.

**Dos reglas nuevas** (fijadas aquí, antes de ver ningún puntaje de los H):
- **Seguro:** se conservan siempre **al menos 2 candidatos sin HatEval**: FINAL_A_quote y el siguiente mejor del 10 por CV. Cubren el caso de que HatEval no se sostenga en la parte privada.
- **Control de fuga (sección 10):** si el puntaje público del primer H sube más de ≈0.02 por encima de lo que predice su CV, **no se selecciona ningún H**.

**Huecos libres.** El puntaje final es el **máximo** de los seleccionados, así que añadir un candidato nunca lo puede bajar: un hueco vacío solo se desperdicia. Si después de a)–c) y del seguro quedan huecos, se rellenan **por CV media** con los no seleccionados, incluidos los que fallan a) o c), salvo los casi duplicados (b). FINAL_A queda para el final.
- La versión anterior de esta regla solo rellenaba con los que fallaban a).
- Se amplió el 2026-10-18, al ver que con los puntajes de los H el criterio c) dejaba 2 huecos vacíos (D45).
- El leaderboard sigue sin usarse para ordenar.

Los envíos antiguos de un solo modelo (FINAL_B y los del 03–04) ya no cumplían a) en el 10 y no se incluyen.

In [19]:
CANDS_10 = {  # candidatos del 10: (pool, método, con regla)
    "FINAL_A_quote": (POOL_D, "stack-all", True), "FINAL_A": (POOL_D, "stack-all", False), "FINAL_C": (POOL_C, "stack-all", False),
    "FINAL_C_quote": (POOL_C, "stack-all", True), "A2_quote": (POOL_V2, "stack-all", True), "D_s10_quote": (POOL_DS10, "stack-all", True),
    "D_ownlabel_quote": (POOL_D, "stack-own", True), "D_weights_quote": (POOL_D, "weights", True), "A_foldbag_quote": (POOL_D, "stack-all", True),
}
FILES_10 = {"FINAL_A_quote": "FINAL_A_quote_ens08__D__stack-all__quote.csv", "FINAL_A": "FINAL_A_ens07__D__stack-all.csv",
            "FINAL_C": "FINAL_C_ens08__C__stack-all.csv", "FINAL_C_quote": "FINAL_C_quote_ens10__C__stack-all__quote.csv",
            "A2_quote": "A2_quote_ens10__v2__stack-all__quote.csv", "D_s10_quote": "D_s10_quote_ens10__Ds10__stack-all__quote.csv",
            "D_ownlabel_quote": "D_ownlabel_quote_ens10__D__stack-own__quote.csv", "D_weights_quote": "D_weights_quote_ens10__D__weights__quote.csv",
            "A_foldbag_quote": "A_foldbag_quote_ens10__D__stack-all__foldbag__quote.csv"}
LB = {"FINAL_A_quote": 0.78598, "FINAL_A": 0.78273, "FINAL_C": 0.77760, "FINAL_C_quote": 0.78019, "A2_quote": 0.78783,
      "D_s10_quote": 0.79035, "D_ownlabel_quote": 0.78714, "D_weights_quote": 0.78450, "A_foldbag_quote": 0.78454}
LB_NEW = {"H5_quote": 0.80331, "H2_quote": 0.80706, "H3_quote": 0.80207,  # puntajes públicos de los H (16–18 oct).
          "H6_quote": 0.80856, "H4_quote": 0.80562, "H1_quote": 0.81235}   # Solo descartan (c) y alimentan la sección 10
for tag, (ids, method, rule) in CANDS_10.items():
    CV[tag] = {fs: crossfit_rule(ids, method, fs, rule) for fs in FOLDSETS}
ALL_FILES = {**FILES_10, **FILES}
SUBS = {tag: pd.read_csv(md.SUBMISSIONS_DIR / f)[LABELS].to_numpy() for tag, f in ALL_FILES.items()}
RULE = {t: (CANDS_10[t][2] if t in CANDS_10 else True) for t in ALL_FILES}
SCORES = {**LB, **LB_NEW}

## 10. Control de fuga

**Qué podría fallar.** El test de la competencia está dentro de HatEval. Si algún tuit del test se hubiera colado en el entrenamiento (un casi duplicado por debajo del umbral, por ejemplo), los H acertarían esos tuits "de memoria":
- su puntaje público subiría **más** de lo que predice la CV;
- en la CV no pasa, porque HatEval se deduplicó también frente a nuestro train.

**Prueba (fijada antes de subir nada):**
- se sube **primero el H con mejor CV**;
- se compara su ganancia en el leaderboard público frente a FINAL_A_quote (0.78598) con su ganancia en la CV (Δ medio de los dos conjuntos de folds);
- si la del leaderboard supera a la de la CV en **más de 0.02**, se avisa y **no se selecciona ningún H**.

0.02 es un margen amplio a propósito: el leaderboard público mide solo una parte del test y diferencias de ±0.005–0.01 entre envíos son ruido (H44).

In [20]:
H_ORDER = sorted(H_TAGS, key=lambda t: -np.mean([CV[t][fs][1]["F1_macro"].mean() for fs in FOLDSETS]))  # sin redondear
FIRST_H = H_ORDER[0]
cv_gain = np.mean([delta(CV[FIRST_H][fs][1], REF[fs])[0]["Δ"] for fs in FOLDSETS])
LEAK = None
if FIRST_H in LB_NEW:
    lb_gain = LB_NEW[FIRST_H] - LB["FINAL_A_quote"]
    LEAK = bool(lb_gain > cv_gain + 0.02)
    print(f"{FIRST_H}: ganancia pública {lb_gain:+.4f} frente a CV {cv_gain:+.4f} -> "
          + ("AVISO: sube mucho más de lo esperado; posible fuga del test. No se selecciona ningún H." if LEAK else
             "coherente con la CV (sin indicios de fuga)."))
else:
    print(f"Primer envío: {FIRST_H} (mejor CV de los H; ganancia esperada en la CV {cv_gain:+.4f}). "
          f"Umbral de aviso: ganancia pública > {cv_gain + 0.02:+.4f}, es decir, puntaje público > {LB['FINAL_A_quote'] + cv_gain + 0.02:.5f}.")

H5_quote: ganancia pública +0.0173 frente a CV +0.0206 -> coherente con la CV (sin indicios de fuga).


## 11. Tabla de candidatos y lista final de 10

In [21]:
SLOTS = 10
pool_tags = [t for t in ALL_FILES if not (LEAK and t.startswith("H"))]
rows = []
for tag in pool_tags:
    main, s2 = CV[tag]["folds"][1], CV[tag]["folds_seed2"][1]
    rows.append({"candidato": tag, "HatEval": "sí" if tag.startswith("H") else "no", "regla": "sí" if RULE[tag] else "no",
                 "CV folds": round(main["F1_macro"].mean(), 4), "CV folds_seed2": round(s2["F1_macro"].mean(), 4),
                 "LB público": SCORES.get(tag, np.nan), "% celdas ≠ FINAL_A_quote": round(100 * (SUBS[tag] != SUBS["FINAL_A_quote"]).mean(), 2)})
TABLE = pd.DataFrame(rows).set_index("candidato")
TABLE["CV media"] = TABLE[["CV folds", "CV folds_seed2"]].mean(axis=1)
TABLE = TABLE.sort_values("CV media", ascending=False)
order = list(TABLE.index)
DIFF = pd.DataFrame([[round(100 * (SUBS[a] != SUBS[b]).mean(), 2) for b in order] for a in order], index=order, columns=order)
with_rule = TABLE[TABLE["regla"] == "sí"]
best_main, best_s2, best_lb = with_rule["CV folds"].max(), with_rule["CV folds_seed2"].max(), max(SCORES.values())
non_h = [t for t in order if not t.startswith("H") and RULE[t] and t != "FINAL_A_quote"]
INSURED = ["FINAL_A_quote", non_h[0]]
admitted, status, only_a = [], {}, []
for tag in order:
    row = TABLE.loc[tag]
    a = best_main - row["CV folds"] <= 0.010 and best_s2 - row["CV folds_seed2"] <= 0.010
    c = np.isnan(row["LB público"]) or row["LB público"] >= best_lb - 0.010
    near = [adm for adm in admitted if DIFF.loc[tag, adm] < 1.0]
    if tag == "FINAL_A":
        status[tag] = "cobertura de la regla (d): solo si sobra hueco"
    elif not c:
        status[tag] = "descartado por c) (LB público > 0.010 bajo el mejor)"
    elif near:
        status[tag] = f"casi duplicado de {near[0]} (b)"
    elif not a:
        status[tag] = "no cumple a) (calidad)"
        only_a.append(tag)
    else:
        admitted.append(tag)
        status[tag] = "admitido"
for tag in INSURED:
    if tag not in admitted:
        status[tag] = status[tag] + " -> seguro sin HatEval"
selected = [t for t in order if t in admitted or t in INSURED]
fill = [t for t in order if t not in selected and t != "FINAL_A" and not status[t].startswith("casi")]  # D45, por CV
for tag in fill[:max(0, SLOTS - len(selected))]:
    selected.append(tag)
    status[tag] += " -> relleno de un hueco libre"
if len(selected) < SLOTS:
    selected.append("FINAL_A")
selected = selected[:SLOTS]
TABLE["estado"] = [status[t] for t in order]
TABLE["seleccionado"] = [t in selected for t in order]
print(f"Mejor CV con la regla: folds {best_main:.4f} | folds_seed2 {best_s2:.4f} | mejor LB público {best_lb:.5f} | "
      f"seguro sin HatEval: {INSURED}" + (" | H excluidos por el control de fuga" if LEAK else ""))
display(TABLE)
print("% de celdas distintas entre los archivos de test (3462 celdas):")
display(DIFF)

Mejor CV con la regla: folds 0.8238 | folds_seed2 0.8286 | mejor LB público 0.81235 | seguro sin HatEval: ['FINAL_A_quote', 'A2_quote']


,HatEval,regla,CV folds,CV folds_seed2,LB público,% celdas ≠ FINAL_A_quote,CV media,estado,seleccionado
candidato,,,,,,,,,
H5_quote,sí,sí,0.8216,0.8275,0.80331,6.79,0.82455,admitido,True
H2_quote,sí,sí,0.8201,0.8283,0.80706,6.87,0.82420,admitido,True
H3_quote,sí,sí,0.8193,0.8286,0.80207,7.02,0.82395,descartado por c) (LB público > 0.010 bajo el mejor) -> relleno de un hueco libre,True
H6_quote,sí,sí,0.8202,0.8272,0.80856,6.93,0.82370,admitido,True
H4_quote,sí,sí,0.8238,0.8233,0.80562,6.07,0.82355,admitido,True
H1_quote,sí,sí,0.8219,0.8251,0.81235,7.39,0.82350,admitido,True
A2_quote,no,sí,0.8072,0.8052,0.78783,3.18,0.80620,descartado por c) (LB público > 0.010 bajo el mejor) -> seguro sin HatEval,True
D_s10_quote,no,sí,0.8069,0.8014,0.79035,2.25,0.80415,descartado por c) (LB público > 0.010 bajo el mejor) -> relleno de un hueco libre,True
FINAL_A_quote,no,sí,0.8060,0.8019,0.78598,0.00,0.80395,descartado por c) (LB público > 0.010 bajo el mejor) -> seguro sin HatEval,True


% de celdas distintas entre los archivos de test (3462 celdas):


,H5_quote,H2_quote,H3_quote,H6_quote,H4_quote,H1_quote,A2_quote,D_s10_quote,FINAL_A_quote,A_foldbag_quote,D_ownlabel_quote,FINAL_C_quote,D_weights_quote,FINAL_A,FINAL_C
H5_quote,0.00,2.92,3.18,2.98,4.36,3.15,6.44,6.96,6.79,6.18,5.98,6.53,6.21,6.93,6.64
H2_quote,2.92,0.00,1.47,1.16,2.77,1.68,6.01,6.70,6.87,6.50,6.18,6.90,7.16,7.02,7.02
H3_quote,3.18,1.47,0.00,1.88,3.44,2.51,6.27,7.02,7.02,6.59,5.92,6.93,7.13,7.16,7.05
H6_quote,2.98,1.16,1.88,0.00,3.29,2.31,6.07,6.82,6.93,6.50,5.89,6.96,6.93,7.08,7.08
H4_quote,4.36,2.77,3.44,3.29,0.00,3.41,5.31,5.78,6.07,5.86,5.78,6.56,6.47,6.21,6.67
H1_quote,3.15,1.68,2.51,2.31,3.41,0.00,6.41,7.05,7.39,6.90,7.05,7.25,7.63,7.54,7.37
A2_quote,6.44,6.01,6.27,6.07,5.31,6.41,0.00,1.96,3.18,3.32,3.64,3.73,4.27,3.32,3.84
D_s10_quote,6.96,6.70,7.02,6.82,5.78,7.05,1.96,0.00,2.25,3.03,3.81,3.38,4.16,2.40,3.50
FINAL_A_quote,6.79,6.87,7.02,6.93,6.07,7.39,3.18,2.25,0.00,2.34,3.52,3.03,4.10,0.14,3.15
A_foldbag_quote,6.18,6.50,6.59,6.50,5.86,6.90,3.32,3.03,2.34,0.00,2.92,3.64,3.32,2.48,3.76


**Interpretación (con los puntajes públicos de los 6 H, 16–18 oct):**
- **Control de fuga superado:** H5_quote, el primero en subirse, ganó +0.0173 en el público frente a +0.0206 en la CV (umbral de aviso: +0.0406).
  - Los 6 H ganan en el público +0.016 a +0.026 frente a FINAL_A_quote, del mismo orden que su CV (+0.019 a +0.021).
  - La distancia entre la CV y el público (≈−0.021 para H5) es la misma que la de los envíos sin HatEval (−0.018 para FINAL_A_quote).
  - Una fuga del test habría hecho que el público subiera **más** que la CV; no pasa.
- **Puntajes públicos:** H1 0.81235 (el mejor del proyecto), H6 0.80856, H2 0.80706, H4 0.80562, H5 0.80331, H3 0.80207.
  - El orden **no** se parece al de la CV: H5 tiene la mejor CV y sale 5.º.
  - El rango entre ellos (0.010) es el mismo que entre los candidatos del 10 con CV casi igual (H68). Es ruido de la parte pública, como estaba previsto (H44).
- **a) Calidad:** pasan los 6 H; ningún candidato del 10 (A2_quote, el mejor, queda 0.017 / 0.023 por debajo).
- **c) Descarte por leaderboard:** el umbral es 0.80235 (0.81235 − 0.010).
  - H3 queda fuera por 0.00028;
  - todos los del 10 quedan fuera, porque el mejor (D_s10_quote, 0.79035) está 0.022 por debajo.
- **Selección (10 de 10):**
  - admitidos: H5, H2, H6, H4 y H1;
  - seguro sin HatEval: A2_quote y FINAL_A_quote;
  - los 3 huecos que quedarían vacíos se rellenan por CV (D45): H3_quote, D_s10_quote y A_foldbag_quote.
  - **Es la misma lista que la provisional (D44).**
- **Quedan fuera:** D_ownlabel_quote, FINAL_C_quote, D_weights_quote, FINAL_A y FINAL_C.

**Plan de subidas.** Quedan el 16, 17 y 18 de octubre y el 19 hasta las 18:00 (hora de Bogotá), con un máximo de 3 envíos por día.
- **Primer envío:** el H con mejor CV, solo, para hacer el control de fuga (sección 10) con su puntaje antes de subir los demás.
- **Después:** el resto de H seleccionados, por orden de CV. Los del 10 ya tienen puntaje público.
- **Ojo:** Kaggle reinicia el contador diario a medianoche UTC, que son las 19:00 en Bogotá.

Cuando llegue cada puntaje público, se añade a `LB_NEW` (celda de la sección 9) y se vuelven a ejecutar las secciones 9–10: el control de fuga y el criterio c) se aplican solos.

In [22]:
to_upload = [t for t in H_ORDER if t in selected and t not in SCORES]
days = ["16 oct", "17 oct", "18 oct", "19 oct (antes de las 18:00)"]
plan = []
for i, tag in enumerate(to_upload):
    plan.append({"día": days[min(i // 3, 3)], "orden": i + 1, "candidato": tag, "archivo": ALL_FILES[tag],
                 "CV media": TABLE.loc[tag, "CV media"],
                 "nota": "PRIMERO, solo: control de fuga" if tag == FIRST_H else "después del control de fuga"})
display(pd.DataFrame(plan).set_index(["día", "orden"]) if plan else "Nada nuevo que subir")
print(f"Selección final propuesta ({len(selected)} de {SLOTS}):")
for i, t in enumerate(selected, 1):
    print(f"  {i}. {t:18s} [{ALL_FILES[t]}] — {status[t]}")

'Nada nuevo que subir'

Selección final propuesta (10 de 10):
  1. H5_quote           [H5_quote_ens11__v2he05__weights__quote.csv] — admitido
  2. H2_quote           [H2_quote_ens11__v2he05__stack-all__quote.csv] — admitido
  3. H6_quote           [H6_quote_ens11__v2he05-new__stack-all__quote.csv] — admitido
  4. H4_quote           [H4_quote_ens11__mix05__stack-all__quote.csv] — admitido
  5. H1_quote           [H1_quote_ens11__v2he10__stack-all__quote.csv] — admitido
  6. A2_quote           [A2_quote_ens10__v2__stack-all__quote.csv] — descartado por c) (LB público > 0.010 bajo el mejor) -> seguro sin HatEval
  7. FINAL_A_quote      [FINAL_A_quote_ens08__D__stack-all__quote.csv] — descartado por c) (LB público > 0.010 bajo el mejor) -> seguro sin HatEval
  8. H3_quote           [H3_quote_ens11__Dhe05__stack-all__quote.csv] — descartado por c) (LB público > 0.010 bajo el mejor) -> relleno de un hueco libre
  9. D_s10_quote        [D_s10_quote_ens10__Ds10__stack-all__quote.csv] — descartado por c) (LB público >

In [23]:
# Registro: los modelos base con HatEval (OOF en folds.csv, sin la regla) y los candidatos H (con la regla)
reg_rows, oof_frames, test_frames = [], [], []
for name in NEEDED:
    pf = single(name, "folds")
    base = NETS[1] if name.endswith("hetext") else name.rsplit("__he", 1)[0]
    found = md.find_model_path(name)
    family = REG.loc[base, "family"] if base in REG.index else REG.loc[base + "__s10", "family"]
    reg_rows.append({"model_id": name, "family": family,
                     "classifier": name.split("__")[1], "variant": name.split("__")[0],
                     "features": f"11: {base} + HatEval ({len(X_EXT)} tuits)", "params": "{}", "notebook": "11",
                     "F1_macro": round(pf["F1_macro"].mean(), 4), "F1_macro_std": round(pf["F1_macro"].std(), 4),
                     "F1_micro": round(pf["F1_micro"].mean(), 4), **{f"F1_{lab}": round(pf[f"F1_{lab}"].mean(), 4) for lab in LABELS},
                     "proba_calibrated": False, "cv_seconds": round(TIMES[(name, "folds")], 1), "saved": found is not None,
                     "size_mb": round(found.stat().st_size / 1e6, 2) if found else None})
    oof_frames.append(md.prediction_frame(name, train["id"], PD["folds"][name], folds))
    test_frames.append(md.prediction_frame(name, test["id"], P_TEST[name]))
for tag, M in MODELS_11.items():
    pf = CV[tag]["folds"][1]
    reg_rows.append({"model_id": M["name"], "family": "ensamble", "classifier": CANDS[tag][1], "variant": "—",
                     "features": f"{tag}: " + " + ".join(M["ids"]), "params": "{}", "notebook": "11",
                     "F1_macro": round(pf["F1_macro"].mean(), 4), "F1_macro_std": round(pf["F1_macro"].std(), 4),
                     "F1_micro": round(pf["F1_micro"].mean(), 4), **{f"F1_{lab}": round(pf[f"F1_{lab}"].mean(), 4) for lab in LABELS},
                     **{f"thr_{lab}": M["model"].thresholds[lab] for lab in LABELS}, "proba_calibrated": True, "saved": True})
    oof_frames.append(md.prediction_frame(M["name"], train["id"], M["oof"], folds, calibrated=True))
    test_frames.append(md.prediction_frame(M["name"], test["id"], M["model"].predict_proba_from_base(M["P_te"]), pred=SUBS[tag],
                                           calibrated=True))
reg = md.update_registry(reg_rows)
md.update_predictions(oof_frames, md.OOF_PATH)
md.update_predictions(test_frames, md.TEST_PRED_PATH)
print(f"Registro: {len(reg)} modelos | añadidos o actualizados: {len(reg_rows)}")

Registro: 235 modelos | añadidos o actualizados: 36


**Interpretación:**
- **Ya están subidos los 6 H y no queda nada nuevo que subir:** los 10 seleccionados tienen puntaje público.
- **Lo único pendiente es marcar en Kaggle los 10 de la lista**, antes del 19 de octubre a las 18:00 (hora de Bogotá).
- **Qué esperar en la parte privada:**
  - el puntaje final será el máximo de los 10;
  - los 6 H están ≈+0.02 por encima de los del 10, tanto en CV como en el público, así que casi con seguridad el máximo será uno de ellos;
  - el seguro sin HatEval solo importaría si la parte privada se comportara muy distinto de la pública y de la CV, de lo que no hay ningún indicio.

## 12. Conclusiones

1. **El corpus de la competencia es un subconjunto de HatEval en español** (todos los ids). Tras deduplicar por id, clave y coseno ≥ 0.85, se conservan **3621 tuits nuevos**; los 1216 que coinciden con el test se descartaron sin leer sus etiquetas.
2. **HatEval está anotado igual que nuestros datos:** acuerdo HS/TR 0.9988 y AG 0.9809, y los desacuerdos en AG son el artefacto de comillas. Pero tiene otra mezcla: tuits más largos y menos positivos.
3. **Todos los modelos mejoran** (14/14, Δ medio +0.016 a +0.020), en la parte baja de lo extrapolado en el 09. El Transformer gana +0.050 y la CNN de palabras +0.04.
4. **Los 6 candidatos H cumplen la regla del 07:** +0.013 a +0.018 (`folds.csv`) y +0.021 a +0.027 (`folds_seed2`) frente a FINAL_A_quote.
5. **El leaderboard público lo confirma, sin indicios de fuga:**
   - +0.016 a +0.026 frente a FINAL_A_quote;
   - mejor envío del proyecto: **H1_quote, 0.81235** (antes, D_s10_quote, 0.79035).
6. **Selección final (10 de 10):** H5, H2, H6, H4, H1, A2_quote, FINAL_A_quote, H3, D_s10_quote y A_foldbag_quote.